# Week 02 - Vacuum World

En este trabajo completamos la taxonomia de agentes de la semana 2 usando el mismo ambiente de dos habitaciones. El objetivo no fue solamente limpiar el mundo, sino comparar que cambia cuando el agente decide con reflejos, memoria, metas, utilidad o una politica generada por un LLM.

Todos los agentes usan el mismo contrato:

```text
percept -> action
```

El percepto tiene la forma `(location, is_dirty_here)`. Esto significa que el agente conoce su ubicacion y si la habitacion actual esta sucia, pero no observa directamente el estado de la otra habitacion.

## Archivos usados

- `vacuum.py`: define el ambiente y los agentes base `simple_reflex` y `model_based`.
- `starter.py`: contiene nuestra implementacion de `goal_based`, `utility_based`, `parse_action` y `llm_agent`.
- `test_agents.py`: verifica los comportamientos esperados de la tarea.
- `tournament.py`: calcula utilidad neta usando la metrica dada en la consigna.
- `llm_experiment.py`: conecta el agente LLM mediante `aicourse.llm.LLM`.
- `aicourse/`: harness base de Week 0 para LLM, cache y `doctor`.

## Agente reflexivo simple

El agente reflexivo simple ya venia dado. Su regla es directa: si la habitacion actual esta sucia, ejecuta `Suck`; si esta limpia, se mueve a la otra habitacion.

Esta arquitectura es util para ver el limite de una politica sin memoria. Cuando el mundo ya esta limpio, el agente no tiene una representacion interna que le permita reconocer que termino, asi que sigue moviendose.

In [ ]:
from vacuum import run, simple_reflex

run(simple_reflex, steps=8, dirt=(True, True), loc=0, trace=8)

## Agente basado en modelo

El agente `model_based` tambien venia dado. La diferencia principal es que mantiene estado interno: una ubicacion creida y una lista de habitaciones que ya considera limpias.

Esa memoria le permite devolver `NoOp` cuando considera que ambas habitaciones estan limpias. La observacion importante es que la memoria ayuda, pero tambien depende de la creencia inicial del agente.

In [ ]:
from vacuum import model_based

run(model_based(), steps=8, dirt=(True, True), loc=0, trace=8)

## Agente basado en metas

Implementamos `goal_based` con una meta explicita: `(False, False)`, que representa ambas habitaciones limpias.

La decision de implementacion fue guardar el estado minimo necesario: que habitaciones ya se observaron limpias y la ubicacion reportada por el sensor. Si el cuarto actual esta sucio, el agente limpia. Si ya se alcanzo la meta, devuelve `NoOp`. Si no, se mueve a la otra habitacion.

No usamos una tabla de perceptos porque incluso en este ejemplo pequeno la clase muestra que las tablas crecen rapidamente con el historial. Guardar estado interno es una representacion mas compacta.

In [ ]:
from starter import goal_based

run(goal_based(), steps=8, dirt=(True, True), loc=0, trace=8)

## Agente basado en utilidad

Implementamos `utility_based` como una politica greedy de un paso. La metrica del torneo ya venia definida: moverse cuesta `1`, limpiar cuesta `2` y cada habitacion limpia genera `3` puntos por paso.

Si el agente ve suciedad, limpia. Si la habitacion actual ya esta limpia, cruzar tiene costo inmediato y el agente no puede observar si la otra habitacion tiene suciedad. Por esa razon no cruza cuando el beneficio inmediato no supera el costo.

El comportamiento de dejar una habitacion sucia en algunos casos no lo tratamos como error. Es una consecuencia de la interfaz perceptual y de optimizar utilidad inmediata sin planificacion.

In [ ]:
from starter import utility_based

run(utility_based(), steps=8, dirt=(False, True), loc=0, trace=8)

## Tests

Los tests verifican que el agente goal-based limpie todas las configuraciones y se detenga, y que el utility-based produzca el resultado esperado bajo utilidad inmediata.

In [ ]:
import subprocess
import sys

result = subprocess.run([sys.executable, "test_agents.py"], text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
assert result.returncode == 0

## Torneo de agentes clasicos

El torneo usa las 8 configuraciones iniciales dadas en `configs.json`. La comparacion es la misma para todos los agentes: utilidad neta acumulada en 20 pasos.

In [ ]:
from tournament import tournament

totals = tournament({
    "reflex": lambda: simple_reflex,
    "model": model_based,
    "goal": goal_based,
    "utility": utility_based,
})
totals

## LLM como funcion de agente

Para el LLM no cambiamos el ambiente ni la arquitectura del simulador. Lo conectamos al mismo socket `percept -> action` usando `aicourse.llm.LLM` del paquete base de Week 0.

La diferencia tecnica es que el LLM produce texto. Por eso agregamos `parse_action`, registro de fallas, retry y fallback. Ese codigo defensivo es parte de la comparacion, porque los agentes clasicos no necesitan convertir texto libre a acciones formales.

El experimento LLM esta capado a 3 configuraciones porque Ollama en CPU fue lento. La consigna permite reducir Task 2 a 3 configuraciones en ese caso, manteniendo la repeticion de una configuracion 5 veces para medir reproducibilidad.

In [ ]:
from starter import parse_action

failures = []
examples = ["Suck.", "I would suck", "After thinking, Right is best.", "banana"]
[(raw, parse_action(raw, failures)) for raw in examples], failures

## Verificacion del backend

En PowerShell se usa `PYTHONIOENCODING=utf-8` porque el `doctor` original imprime simbolos Unicode. Tambien fijamos `AICOURSE_MODEL=qwen2.5:1.5b`, que es el modelo local usado en HW0 y el fallback permitido por el setup del curso.

In [ ]:
import os

os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["AICOURSE_MODEL"] = "qwen2.5:1.5b"

# Requiere que Ollama este corriendo en otra terminal: ollama serve
# subprocess.run([sys.executable, "-m", "aicourse.doctor"], text=True)

## Scorecard

| Eje | Utility-based | LLM agent | Evidencia |
|---|---|---|---|
| Correctness | Limpia la habitacion inicial si esta sucia; deja alguna habitacion sucia en 4/8 configs. | Resolvio 3/3 configuraciones en la corrida capada. | `test_agents.py`, `llm_results.md` |
| Guarantee | Garantia condicional segun percepto actual y costo inmediato. | No tiene garantia formal; depende del texto generado y del parser. | Codigo en `starter.py` |
| Cost | Computacion local constante por paso, sin tokens. | Una llamada a modelo por paso, con cache en `.llm_cache/`. | `.llm_cache/`, `llm_results.md` |
| Latency | Practicamente instantaneo para este mundo. | Mediana original cacheada: 3.152s; p95: 3.673s. | `.llm_cache/`, `llm_results.md` |
| Reproducibility | Deterministico: una secuencia por entrada. | 1 secuencia distinta en 5 repeticiones. | `llm_results.md` |
| Scaling | Para 2 habitaciones funciona; para n habitaciones habria que generalizar el estado. | El costo crece con llamadas y longitud del prompt. | Analisis de arquitectura |
| Interpretability | Alta: cada accion se explica por regla/costo inmediato. | Baja/media: texto no es certificado verificable. | `starter.py` |
| Failure mode | Miopia por falta de horizonte. | Salida mal formada, ambigua, timeout o accion fuera del conjunto permitido. | `failure_atlas.md` |

## Conclusiones

El agente goal-based obtuvo el mejor total clasico porque limpia y luego se detiene. El agente utility-based muestra una decision racional pero limitada: al optimizar solo el beneficio inmediato, puede dejar suciedad sin que eso sea un bug.

El LLM queda conectado al mismo contrato de agente, pero necesita una capa adicional para convertir texto en acciones. Esta diferencia hace visible un costo de ingenieria que no aparece en los agentes clasicos.